In [ ]:
import numpy as np
import torch
import pytorch_lightning as pl

from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Normalize, Resize, ToTensor, CenterCrop
from torch.utils.data import random_split
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

In [ ]:
class FlowerDataModel(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size=32, split_ratio=0.8):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.split_ratio = split_ratio

    def setup(self, stage=None):
        transform = Compose([
            Resize(64),
            CenterCrop(64),
            ToTensor(),
            Normalize(mean=[0.4882, 0.4374, 0.3103], std=[0.2832, 0.2543, 0.2812])
        ])
        dataset = ImageFolder(root=self.data_dir, transform=transform)
        train_size = int(self.split_ratio * len(dataset))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

In [ ]:
class FlowerClassifier(pl.LightningModule):
    def __init__(self, num_classes=5, num_filters=16, dropout_rate=0, learning_rate=3e-4):
        super().__init__()
        self.conv1 = nn.Conv2d(3, num_filters, kernel_size=3)
        self.conv2 = nn.Conv2d(num_filters, num_filters * 2, kernel_size=3)
        self.conv3 = nn.Conv2d(num_filters * 2, num_filters * 4, kernel_size=3)
        self.fc1 = nn.Linear(num_filters * 4 * 6 * 6, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.p = dropout_rate
        self.lr = learning_rate

        # Store training and validation losses for plotting
        self.training_losses = []
        self.validation_losses = []

    def featurizer(self, x):
        # First convolutional block
        # 3@64x64 -> n_filters@62x62 -> n_filters@31x31
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Second convolutional block
        # n_filters@31x31 -> n_filters@29x29 -> n_filters@14x14
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Third convolutional block
        # n_filters@14x14 -> n_filters@12x12 -> n_filters@6x6
        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Input dimension (n_filters*4@6x6)
        # Output dimension (n_filters * 4 * 6 * 6)
        x = nn.Flatten()(x)
        return x
    
    def classifier(self, x):
        # Input dimension (n_feature * 4 * 6 * 6)
        # Output dimension (128)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc1(x)
        x = F.relu(x)
        # Input dimension (128) -> Output dimension (64)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc2(x)
        x = F.relu(x)
        # Input dimension (64) -> Output dimension (5)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc3(x)
        return x
    
    def forward(self, x):
        x = self.featurizer(x)
        x = self.classifier(x)
        return x
    
    def training_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.training_losses.append(loss.item())

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)
        acc = (outputs.argmax(dim=1) == labels).float().mean()
        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('val_acc', acc, on_step=True, on_epoch=True, prog_bar=True)
        self.validation_losses.append(loss.item())

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.lr)

In [ ]:
data_dir = "./Datasets/flower_photos"
data_module = FlowerDataModel(data_dir=data_dir, batch_size=64, split_ratio=0.8)
model = FlowerClassifier(dropout_rate=0.2)
trainer = Trainer(max_epochs=100, devices="auto", accelerator="auto", log_every_n_steps=10)